In [10]:

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [11]:
spark = (( SparkSession
            .builder
            .appName('supplier')
            .config('spark.jar', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
        ))

In [12]:
#loding data from table
tmp_supplier = (( spark
                    .read
                    .format('jdbc')
                    .option('url', mssqlsource)
                    .option('driver', mssqldriver)
                    .option('user', mssqluser)
                    .option('password', mssqlpwd)
                    .option('query', 
                            """ 
                                select
                                
                                    supplierid,
                                    suppliercategoryid,
                                    deliverymethodid,
                                    supplierreference,
                                    bankinternationalcode,
                                    suppliername,
                                    bankaccountname,
                                    bankaccountbranch,
                                    paymentdays,
                                    primarycontactpersonid,
                                    alternatecontactpersonid,
                                    deliverycityid,
                                    postalcityid,
                                    deliverypostalcode,
                                    deliveryaddressline1,
                                    deliveryaddressline2
                                    
                                from purchasing.suppliers
                            
                            """))
                ).load()

In [13]:
tmp_categorysupplier = (( spark
                            .read
                            .format('jdbc')
                            .option('url', mssqlsource)
                            .option('driver', mssqldriver)
                            .option('user', mssqluser)
                            .option('password', mssqlpwd)
                            .option('query', 
                                    
                                    """ 
                                    
                                        select 
                                        
                                            suppliercategoryid,
                                            suppliercategoryname
                                            
                                        from purchasing.suppliercategories
                                    
                                    """
                                    )
                            )).load()

In [14]:
#joining dataframes

join_supplier = (( tmp_supplier
                  
                  #joining by suppliercategoryid
                  #since columns with datetime information about elt have the same value,
                  #i'll keep them out of this dataframe
                  
                    .join(tmp_categorysupplier,
                          'suppliercategoryid',
                          'left')
                  ))

In [15]:
#removing nulls
#fill empty values

renamed_column = (( join_supplier
                    .select('*',
                                f.coalesce('deliverymethodid', f.lit(999))
                                    .alias('deliverymethodid_n'),
                                
                                #adjust some fields that's empty
                                f.when(join_supplier.deliveryaddressline1 == '', 'Unknown')
                                 .otherwise(join_supplier.deliveryaddressline1)
                                    .alias('deliveryaddress_n'),
                                    
                                #removing comma character from column
                                f.regexp_replace('suppliername', '[^A-Za-z0-9]', ' ')
                                    .alias('suppliername_n')
                    )
                ))


In [16]:
#rename and drop columns

supplier_frame = (( renamed_column
                    .drop('deliverymethodid')
                    .drop('deliveryaddressline1')
                    .drop('suppliername')
                    
                    #renaming column
                    .withColumnRenamed('deliverymethodid_n', 'deliverymethodid')
                    .withColumnRenamed('deliveryaddress_n', 'deliveryaddress')
                    .withColumnRenamed('suppliername_n', 'suppliername')
                    .withColumnRenamed('supplierreference', 'suppliercode')
                    .withColumnRenamed('primarycontactpersonid', 'contact_id')
                    .withColumnRenamed('alternatecontactpersonid', 'submanager_contactid')
                    .withColumnRenamed('bankaccountname', 'bankname')
                    .withColumnRenamed('bankaccountbranch', 'bank_branch')
                    .withColumnRenamed('deliverypostalcode', 'postalcode')
                    .withColumnRenamed('deliveryaddressline2', 'optional_address')
                    .withColumnRenamed('deliverycityid', 'city_id')
                ))

In [17]:
#reorganizing columns

insert_supplier = (( supplier_frame
                      .select(
                              'supplierid',
                              'suppliercategoryid',
                              'suppliercode',
                              'deliverymethodid',
                              'suppliername',
                              'bankinternationalcode',
                              'bankname',
                              'bank_branch',
                              'contact_id',
                              'submanager_contactid',
                              'paymentdays',
                              'city_id',
                              'postalcode',
                              'deliveryaddress',
                              'optional_address'
                            )
))

In [18]:
#write to table

(( insert_supplier
    .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('driver', mssqldriver)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('dbtable', 'dimension.supplier')
    .save(mode= 'overwrite')
))

24/04/15 15:42:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
